# 7주차 ③ 데이터 증강과 혼합 정밀도(AMP) — 실습 8~11  〔교수용 정답본〕

> ⚠️ **이 파일은 교수용입니다.** 학생에게는 `13_augment_amp_blank.ipynb` 를 배포하세요.
> 빈칸 7곳 · 15줄이 모두 채워져 있습니다.

---

## 데이터 증강 — 데이터를 늘리지 않고 늘린다

2교시 학습 곡선에서 **훈련 손실은 내려가는데 검증 정확도는 멈추는** 것을 봤습니다.
**과적합**입니다. 6주차의 답은 Dropout·BatchNorm·가중치 감쇠였습니다.
오늘은 **네 번째 답**입니다.

```
        원본 고양이 사진 1장
              │
   ┌──────────┼──────────┬──────────┐
   ▼          ▼          ▼          ▼
 조금 잘라   좌우 뒤집어  밝기 바꿔   ...
   │          │          │          │
   └──────────┴──────────┴──────────┘
        모델 입장에서는 매 epoch 처음 보는 사진
```

> **핵심 ★★**: 증강은 **저장된 파일을 늘리는 게 아닙니다.**
> `DataLoader` 가 이미지를 꺼낼 때마다 **그 자리에서 무작위로 변형**합니다.
> 그래서 같은 사진을 10 epoch 돌면 **10번 다 다른 사진**입니다.

| 증강 | 무엇 | CIFAR-10 에 쓰나 |
|---|---|---|
| `RandomCrop(32, padding=4)` | 4칸 덧대고 32×32 로 무작위 자르기 | ★★ **거의 항상** |
| `RandomHorizontalFlip()` | 좌우 뒤집기 | ★★ **거의 항상** |
| `ColorJitter(...)` | 밝기·대비 흔들기 | ★ 때때로 |
| `RandomVerticalFlip()` | 위아래 뒤집기 | ✗ **쓰면 안 됨** — 뒤집힌 자동차는 없다 |

> **판단 기준 ★**: *"실제로 그런 사진이 있을 법한가?"*
> 숫자 인식에서 좌우 뒤집기를 하면 안 되는 이유도 같습니다 — **뒤집힌 2 는 2 가 아닙니다.**

### `transforms` 와 `transforms.v2`

오늘부터 **`v2`** 를 씁니다. 기능은 같고 **더 빠르며, 박스·마스크도 같이 변환**해 줍니다(10주차 객체 탐지).
`v2` 에서는 `ToTensor()` 대신 **`ToImage()` + `ToDtype(torch.float32, scale=True)`** 두 줄을 씁니다.

---

## 실습 8 — 증강을 눈으로 확인한다 (8분)

**먼저 보고 나서 학습에 넣습니다.** 증강은 눈으로 확인하지 않으면 실수를 놓칩니다.

In [ ]:
# 셀 1 — 증강 파이프라인 만들기
import torch, torch.nn as nn, time, os
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    N_TRAIN, N_VAL, EPOCHS = 45000, 5000, 15
else:
    N_TRAIN, N_VAL, EPOCHS = 10000, 2000, 8
print("device :", device, f"| 훈련 {N_TRAIN}장 / {EPOCHS} epoch")

MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_tf = v2.Compose([
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

test_tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])

print("train_tf 단계 수 :", len(train_tf.transforms))
print("test_tf  단계 수 :", len(test_tf.transforms), " ← 증강이 없어서 더 짧다")

In [ ]:
# 셀 2 — 같은 사진에 증강을 8번 적용해 본다
raw = datasets.CIFAR10("data", train=True, download=True)      # transform 없이 원본
CLASSES = ["비행기","자동차","새","고양이","사슴","개","개구리","말","배","트럭"]
img, label = raw[7]

def 되돌리기(t):                       # 정규화를 풀어야 사람 눈에 보인다
    m = torch.tensor(MEAN).view(3,1,1); s = torch.tensor(STD).view(3,1,1)
    return (t * s + m).clamp(0, 1).permute(1, 2, 0)

fig, ax = plt.subplots(1, 9, figsize=(15, 2.1))
ax[0].imshow(img); ax[0].set_title("원본"); ax[0].axis("off")
for i in range(1, 9):
    ax[i].imshow(되돌리기(train_tf(img))); ax[i].set_title(f"증강 {i}"); ax[i].axis("off")
fig.suptitle(f"{CLASSES[label]} — 같은 사진, 매번 다른 입력"); plt.tight_layout(); plt.show()

# 검증용은 몇 번을 해도 같은가?
a, b = test_tf(img), test_tf(img)
print("test_tf 두 번 적용이 완전히 같은가 :", torch.equal(a, b))

> **관찰 포인트 ★★**: 8장이 **전부 다릅니다.** 위치가 밀리고, 좌우가 바뀌고, 밝기가 다릅니다.
> 모델은 매 epoch **처음 보는 사진**을 받는 셈입니다.

> **핵심 ★★**: `test_tf` 두 번 적용이 **완전히 같다(True)** 는 것을 확인하세요.
> **검증·테스트에는 증강을 넣지 않습니다.**
> 넣으면 같은 모델인데 **잴 때마다 점수가 달라져서** 비교가 불가능해집니다.
> 6주차에서 *"평가 조건은 고정한다"* 고 한 것과 같은 이야기입니다.

> **함정 ★**: `v2.Normalize` 는 **`ToDtype(scale=True)` 뒤에** 와야 합니다.
> 순서를 바꾸면 0~255 정수에 정규화를 적용하게 되어 값이 엉뚱해집니다.

---

## 실습 9 — 증강을 넣고 다시 학습 ★ (10분)

2교시와 **완전히 같은 모델·같은 epoch**, 증강만 추가합니다.

In [ ]:
# 셀 3 — 훈련에만 증강, 검증에는 증강 없이
full_aug   = datasets.CIFAR10("data", train=True, transform=train_tf)
full_plain = datasets.CIFAR10("data", train=True, transform=test_tf)
test_set   = datasets.CIFAR10("data", train=False, transform=test_tf)
rest = len(full_aug) - N_TRAIN - N_VAL

# 같은 시드를 주므로 몇 번을 불러도 같은 자리에서 나뉜다 (제공)
def 쪼개기(dataset):
    return random_split(dataset, [N_TRAIN, N_VAL, rest],
                        generator=torch.Generator().manual_seed(42))

train_set, _, _ = 쪼개기(full_aug)      # 훈련 — 증강 O
_, val_set,  _  = 쪼개기(full_plain)    # 검증 — 증강 X

train_plain, _, _ = 쪼개기(full_plain)   # 격차 측정용 (증강 없는 훈련 데이터)

train_loader       = DataLoader(train_set,   batch_size=128, shuffle=True)
train_plain_loader = DataLoader(train_plain, batch_size=256, shuffle=False)  # 격차 측정용
val_loader         = DataLoader(val_set,     batch_size=256, shuffle=False)
test_loader        = DataLoader(test_set,    batch_size=256, shuffle=False)
print(f"훈련 {len(train_set)}장(증강 O) / 검증 {len(val_set)}장(증강 X) / 테스트 {len(test_set)}장")

In [ ]:
# 셀 4 — 2교시와 같은 모델을 처음부터 다시 학습
class CNN(nn.Module):                     # ★ 2교시와 완전히 같은 구조여야 불러올 수 있다
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(self.block3(self.block2(self.block1(x))))


criterion = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(m, loader):
    m.eval(); run = 0.0; c = t = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = m(x)
        run += criterion(out, y).item() * y.size(0)
        c += (out.argmax(1) == y).sum().item(); t += y.size(0)
    return run / t, c / t


# ── 2교시 결과(증강 없음)를 불러와 기준선으로 쓴다
base = CNN().to(device)
base.load_state_dict(torch.load("models/cnn_cifar10.pt", map_location=device))
기준_훈련손실, _ = evaluate(base, train_plain_loader)   # ★ 증강 없는 훈련 데이터로 잰다
기준_검증손실, 기준_검증 = evaluate(base, val_loader)
_, 기준_테스트 = evaluate(base, test_loader)
print(f"[증강 X · 2교시] 훈련손실 {기준_훈련손실:.4f} / 검증손실 {기준_검증손실:.4f} "
      f"/ 검증 {기준_검증*100:.2f}% / 테스트 {기준_테스트*100:.2f}%")

# ── 증강을 넣고 다시 학습
torch.manual_seed(42)
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

aug_hist = []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        running += loss.item() * y.size(0)
    scheduler.step()
    vl, acc = evaluate(model, val_loader); aug_hist.append(acc)
    print(f"epoch {epoch:2d} | 훈련 손실 {running/len(train_set):.4f} | 검증 손실 {vl:.4f} "
          f"| 검증 {acc*100:5.2f}% | {time.time()-t0:5.1f}s")

증강_훈련손실, _ = evaluate(model, train_plain_loader)   # ★ 기준선과 똑같은 데이터로 잰다
증강_검증손실, _ = evaluate(model, val_loader)
_, 증강_테스트 = evaluate(model, test_loader)

print()
print("                 훈련손실   검증손실   격차     테스트 정확도")
print(f"증강 X (2교시)   {기준_훈련손실:7.4f}   {기준_검증손실:7.4f}   "
      f"{기준_검증손실-기준_훈련손실:6.4f}   {기준_테스트*100:6.2f}%")
print(f"증강 O (지금)    {증강_훈련손실:7.4f}   {증강_검증손실:7.4f}   "
      f"{증강_검증손실-증강_훈련손실:6.4f}   {증강_테스트*100:6.2f}%")
print()
print(f"과적합 격차(검증손실 − 훈련손실) : "
      f"{기준_검증손실-기준_훈련손실:.4f} → {증강_검증손실-증강_훈련손실:.4f}")
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/cnn_cifar10_aug.pt")

> **관찰 포인트 ★★ — 정확도가 아니라 "격차"를 보세요.**
>
> 짧게 돌리면 **증강 쪽 정확도가 오히려 낮게** 나옵니다. 정상입니다.
> 매번 다른(더 어려운) 사진을 보니 **아직 덜 배운 상태**인 것뿐입니다.
>
> 대신 마지막 표의 **`격차`(검증손실 − 훈련손실)** 를 보세요. **증강 쪽이 훨씬 작습니다.**
> 격차가 크다는 건 **훈련 데이터를 외웠다**는 뜻이고, 그게 과적합입니다.
> **증강은 외우는 것을 막습니다** — 그 효과가 정확도로 나타나려면 **더 오래 돌려야** 합니다.

> **정직하게 ★**: 증강은 모델이 **더 오래 학습할 수 있게** 해 주는 기법입니다.
> 8 epoch 은 *"아직 외울 단계도 아닌데 문제만 어려워진"* 구간입니다.
> **테스트 정확도 차이가 음수여도 당황하지 마세요** — 격차가 줄었으면 증강은 제 일을 한 것입니다.
> **epoch 를 늘리면 정확도도 역전됩니다.** (과제에서 확인할 항목입니다.)

> **막히면**: `models/cnn_cifar10.pt` 가 없다고 하면 2교시 노트북 셀 4 를 실행하세요.

---

## 혼합 정밀도(AMP) — 같은 GPU 로 더 빨리

지금까지 모든 숫자는 **float32**(32비트)였습니다.
**일부 연산만 16비트로** 바꾸면 어떻게 될까요?

| | float32 | float16 |
|---|---|---|
| 메모리 | 4바이트 | **2바이트** (절반) |
| 속도 | 기준 | **2~3배** (Tensor Core 가 있는 GPU) |
| 표현 범위 | 넓다 | **좁다** — 아주 작은 값은 **0 이 된다** ★ |

```
   "혼합"(mixed) 인 이유

   합성곱·행렬곱   →  float16  (빠름. 정밀도가 덜 중요)
   합·평균·손실    →  float32  (정확해야 함)
                      ↑ PyTorch 가 자동으로 나눠 준다  = autocast
```

### 왜 GradScaler 가 필요한가 ★★

기울기는 **아주 작은 값**입니다. float16 으로 표현하면 **0 으로 사라집니다(언더플로)**.
0 이 된 기울기로는 학습이 안 됩니다.

```
   손실 × 1024  →  역전파  →  기울기도 1024배  →  0 이 안 됨
                                    │
                            갱신 직전에 ÷1024 로 되돌린다
```
이 일을 자동으로 해 주는 것이 **`GradScaler`** 입니다. 배율도 알아서 조절합니다.

---

## 실습 10 — AMP 적용과 시간 측정 ★ (11분)

**바뀌는 줄은 세 군데뿐**입니다. 학습 루프의 뼈대는 그대로입니다.

```python
scaler = torch.amp.GradScaler(device.type)               # ① 준비

with torch.autocast(device_type=device.type, dtype=...): # ② 순전파를 감싼다
    out = model(x); loss = criterion(out, y)

optimizer.zero_grad()
scaler.scale(loss).backward()                            # ③ 손실을 키워서 역전파
scaler.step(optimizer)                                   #    갱신 (안에서 되돌림)
scaler.update()                                          #    배율 조정
```

In [ ]:
# 셀 5 — 같은 구간을 fp32 로 한 번, AMP 로 한 번 재 본다
amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
print(f"AMP dtype : {amp_dtype}   (GPU 는 float16, CPU 는 bfloat16 이 기본)")

N_ITER = 30                                  # 30 배치만 재면 충분하다

def 재보기(use_amp):
    torch.manual_seed(0)
    m = CNN().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)

    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    m.train()
    it = iter(train_loader)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()   # ★ 실행마다 초기화해야 둘을 비교할 수 있다
    t0 = time.time()
    for _ in range(N_ITER):
        x, y = next(it)
        x, y = x.to(device), y.to(device)

        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            loss = criterion(m(x), y)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

    if device.type == "cuda":
        torch.cuda.synchronize()             # ★ GPU 는 비동기라 이걸 빼면 엉터리 시간이 나온다
    peak = torch.cuda.max_memory_allocated() / 1024**2 if device.type == "cuda" else 0.0
    return time.time() - t0, loss.item(), peak


t32, l32, m32 = 재보기(False)
tam, lam, mam = 재보기(True)
print(f"\nfp32 : {t32:6.2f}초 ({N_ITER} iter) | 마지막 손실 {l32:.4f}")
print(f"AMP  : {tam:6.2f}초 ({N_ITER} iter) | 마지막 손실 {lam:.4f}")
print(f"속도  : {t32/tam:.2f}배")
if device.type == "cuda":
    print(f"VRAM  : fp32 {m32:.0f} MB → AMP {mam:.0f} MB   ({m32/max(mam, 1e-9):.2f}배 절약)")
else:
    print("VRAM  : CPU 라서 측정하지 않는다 (GPU 에서만 의미가 있다)")

> **관찰 포인트 ★**: **손실 값이 거의 같습니다.** 정확도를 버리지 않았다는 뜻입니다.

> **정직하게 ★★ — CPU 에서는 빨라지지 않습니다.**
> AMP 는 **GPU 의 Tensor Core** 를 쓰기 위한 기법입니다.
> CPU 에는 그 하드웨어가 없어서 **1배 근처이거나 오히려 느립니다.**
> 집에서 CPU 로 돌렸는데 안 빨라졌다면 **정상입니다.**
> 실습실 GPU 에서는 보통 **1.5~3배**, 그리고 **메모리가 절반**이라
> 배치 크기를 두 배로 키울 수 있습니다.

> **함정 ★**: GPU 시간을 잴 때 `torch.cuda.synchronize()` 를 빼면
> **말도 안 되게 빠른 숫자**가 나옵니다. GPU 작업은 비동기라 명령만 던지고 바로 돌아오기 때문입니다.

> **AMP 를 쓰지 말아야 할 때**: 데이터가 아주 작아 GPU 가 놀고 있을 때는 효과가 없습니다.
> **병목이 GPU 연산일 때만** 의미가 있습니다.

---

## 실습 11 — GradScaler 가 없으면 무슨 일이 생기나 (4분)

말로 들은 **언더플로**를 숫자로 확인합니다. 학습은 하지 않습니다 — **2줄이면 됩니다.**

In [ ]:
# 셀 6 — 작은 기울기를 float16 에 넣어 본다
기울기 = torch.tensor([1e-9, 5e-9, 2e-8, 1e-7, 3e-6, 2e-4, 0.5])

def 보기(t):                      # 17자리 실수는 읽기 어려우니 다듬어서 본다
    return "[" + ", ".join("0" if v == 0 else f"{v:.2e}" for v in t.tolist()) + "]"

print("원래 값(fp32) :", 보기(기울기))

그냥 = 기울기.half()

print("fp16 로 그냥  :", 보기(그냥))

죽은비율 = (그냥 == 0).float().mean().item()

print(f"0 이 되어 사라진 비율 : {죽은비율*100:.1f}%   ← 이만큼은 학습에 반영되지 않는다")

# GradScaler 가 하는 일 : 키웠다가 되돌린다
배율 = 1024.0
살린 = (기울기 * 배율).half().float() / 배율
print("\n1024배 → fp16 → ÷1024 :", 보기(살린))
print("0 이 된 비율 :", f"{(살린 == 0).float().mean().item()*100:.1f}%")

> **관찰 포인트 ★★**: 그냥 fp16 으로 바꾸면 **작은 기울기들이 0 이 됩니다.**
> 1024배 해서 넣었다가 되돌리면 **살아남습니다.** 이것이 `GradScaler` 의 전부입니다.
>
> 실제 학습에서 기울기의 상당수는 이 정도로 작습니다.
> **그래서 fp16 + GradScaler 없음 = 학습이 안 됨** 입니다.

> **왜 이 값들인가 ★**: fp16 이 표현할 수 있는 가장 작은 수는 **약 5.96e-8**(비정규수)입니다.
> 그보다 작은 `1e-9`, `5e-9`, `2e-8` 은 **0 으로 내려앉습니다.**
> 반대로 `1e-7` 부터는 살아남습니다 — 경계가 어디인지 값을 바꿔 가며 확인해 보세요.

> **참고**: CPU 의 기본값인 **bfloat16** 은 사정이 다릅니다.
> 정밀도(유효숫자)는 fp16 보다 나쁘지만 **표현 범위는 fp32 와 같아서** 언더플로가 없습니다.
> 그래서 bfloat16 을 쓰면 **GradScaler 가 없어도 됩니다.**
> 최신 GPU(A100·H100 등)에서 bf16 을 선호하는 이유입니다.

---

## 3교시 핵심 정리

| 개념 | 한 줄 정의 |
|------|-----------|
| **데이터 증강** ★★ | 꺼낼 때마다 무작위 변형. **파일이 늘지 않는다** |
| **증강 고르는 법** ★ | *"실제로 그런 사진이 있을 법한가?"* |
| **검증·테스트** ★★ | **증강 금지.** 조건이 고정돼야 비교가 된다 |
| **`transforms.v2`** | `ToImage()` + `ToDtype(float32, scale=True)` |
| **AMP** | 무거운 연산만 16비트. **정확도 유지, 속도·메모리 이득** |
| **`autocast`** | 어떤 연산을 16비트로 할지 **자동으로 나눠 준다** |
| **`GradScaler`** ★★ | 손실을 키워 역전파 → **작은 기울기가 0 이 되는 것을 막는다** |
| **CPU 에서는** ★ | AMP 가 빨라지지 않는다. **GPU 전용 기법** |

### 3교시 체크리스트

- [ ] 증강이 **파일을 늘리는 게 아니라는 것**을 안다 ★★
- [ ] 8장이 서로 다른 것을 눈으로 봤다
- [ ] **검증에 증강을 넣으면 안 되는 이유**를 말할 수 있다 ★★
- [ ] 증강 유/무 정확도를 **직접 비교**했다
- [ ] AMP 3줄(`GradScaler` · `autocast` · `scaler.scale/step/update`)을 안다 ★
- [ ] **`GradScaler` 가 왜 필요한지** 숫자로 확인했다 ★★
- [ ] CPU 에서 AMP 가 안 빨라지는 이유를 안다

### 저장과 제출

`13_augment_amp.ipynb` 로 저장합니다. **과제 제출물입니다.**

---

## 1~7주차 되돌아보기 — 중간고사 범위

```
  1주   딥러닝이란 / 특징을 사람이 설계하지 않는다
  2주   개발 환경 · 텐서
  3주   텐서 연산 · 브로드캐스팅 · GPU
  4주   자동미분 · 계산그래프 · 경사하강법
  5주   퍼셉트론 → MLP · 활성함수 · 기울기 소실 · 학습 루프 5단계
  6주   옵티마이저 · 스케줄러 · 과적합 · 평가지표 · 재현성 · 체크포인트
  7주   합성곱 · 풀링 · CNN 구조 · 데이터 증강 · AMP        ← 오늘
```

**관통하는 하나의 줄기**: *"특징을 사람이 만들지 않고 데이터에서 배운다"*
— 1주차의 선언이 오늘 **특징맵**으로 눈앞에 나타났습니다.

> **다음 주 중간고사**입니다. 오늘 만든 노트북 3개를 다시 훑어보는 것이
> 가장 빠른 복습입니다. 특히 **출력 크기 공식**과 **학습 5단계**는 손으로 쓸 수 있어야 합니다.